In [0]:
df = (
    spark.readStream
        .format("cloudFiles") 
        .option('cloudFiles.format', 'json')
        .option('cloudFiles.schemaLocation', '/Volumes/gizmobox/landing/operational_data/customers_auto_loader/_schema')
        .option('cloudFiles.inferColumnTypes', 'true')
        .option('cloudFiles.schemaHints', 'date_of_birth DATE, member_since DATE, created_timestamp TIMESTAMP')
        .option('cloudFiles.schemaEvalutionMode', 'addNewColumns')
        .load("/Volumes/gizmobox/landing/operational_data/customers_auto_loader/")
)
    

In [0]:
from pyspark.sql.functions import current_timestamp, col
customer_transformed_df = (
                            df.withColumn('file_path', col('_metadata.file_path'))
                              .withColumn('ingestion_date', current_timestamp())
)

In [0]:
streaming_customer = (
                        customer_transformed_df.writeStream
                                                .format('delta')
                                                .option('checkpointLocation', '/Volumes/gizmobox/landing/operational_data/customers_auto_loader/_checkpoint_streaming')
                                                .option('mergeSchema', 'true')
                                                .toTable('gizmobox.bronze.customers_auto_loader')
)

In [0]:
%sql
drop table gizmobox.bronze.customers_auto_loader

In [0]:
%sql
select * from gizmobox.bronze.customers_auto_loader

In [0]:
streaming_customer.stop()